# Amit, Gutfreund & Sompolinsky 1985：论文复现

**a 方程/模式输入 → b 求解或构造 → c 选择物理解 → d 动力学/能量 → e 测量阈值 → f 展示**

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Heptazero/nn-labs/blob/main/hopfield-1985/hopfield_1985.ipynb)

## 0. 这篇快报到底有什么可复现结果？

这篇 4 页 PRL 是 replica mean-field 理论快报，不包含现代意义上的数据集实验、随机重复或误差条。原文只有两张图：

|原文对象|复现内容|论文位置|证据性质|
|---|---|---|---|
|Fig. 1|零温 FM 记忆态的平均错误百分比随载荷率变化|式 (9)–(12)，p.1532|平均场方程的理论曲线|
|Fig. 2|$T_M$、$T_C$ 与 $T_R$ 随载荷率变化的有限温度相图|式 (5)–(8)、(13)，p.1532–1533|RS 鞍点与稳定性方程的数值解|

另外保留两个“原文公式核对”，但不再把它们冒充成第三、第四个实验：FM/SG 的零温能量交叉，以及式 (12) 的精确存储渐近尺度。最后的有限 $N$ 异步动力学明确标为本 notebook 新增机制检查，不是论文图。

论文：D. J. Amit, H. Gutfreund, H. Sompolinsky, *Physical Review Letters* 55, 1530–1533 (1985), [DOI: 10.1103/PhysRevLett.55.1530](https://doi.org/10.1103/PhysRevLett.55.1530)。

### 复现边界与运行方式

这里的“运行”是对平均场积分、鞍点方程和稳定性条件做数值求解，不是训练神经网络，也不是对论文未提供的数据重新做统计实验。前两个主节严格对应原文 Fig. 1 与 Fig. 2；后面的公式核对和有限网络机制检查单独分层。

请在 Colab 中选择 **Runtime → Restart session and run all**。Notebook 不执行 `pip install`，只使用 Colab 自带的 NumPy、SciPy、pandas 与 Matplotlib。修改公共组件后必须重新运行全部单元，避免旧变量污染后续结论。

## 1. 公共实验组件

### 1.1 环境、结果对象与随机数

In [ ]:
from dataclasses import dataclass
from pathlib import Path
from urllib.request import urlretrieve

import matplotlib.font_manager as fm
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.optimize import least_squares
from scipy.special import erf

plt.style.use("seaborn-v0_8-whitegrid")

# 中文字体补丁：只会在 Colab 运行时下载字体，不需要安装 Python 环境
_FONT_PATH = Path("NotoSansCJKtc-Regular.otf")
_FONT_URL = (
    "https://raw.githubusercontent.com/notofonts/noto-cjk/main/"
    "Sans/OTF/TraditionalChinese/NotoSansCJKtc-Regular.otf"
)
if not _FONT_PATH.exists():
    try:
        urlretrieve(_FONT_URL, _FONT_PATH)
    except OSError as error:
        print(f"中文字体下载失败：{error}")
if _FONT_PATH.exists():
    fm.fontManager.addfont(_FONT_PATH)
    plt.rcParams["font.family"] = "Noto Sans CJK TC"
plt.rcParams["axes.unicode_minus"] = False

SEED = 1985


@dataclass(frozen=True)
class ZeroTemperatureBranch:
    alpha: np.ndarray         # (K,) -- 载荷率 p/N
    overlap: np.ndarray       # (K,) -- 与目标模式的宏观重叠 m
    response: np.ndarray      # (K,) -- C = beta(1-q) 的零温极限
    noise: np.ndarray         # (K,) -- 随机重叠强度 r
    error_percent: np.ndarray # (K,) -- 100 * (1-m) / 2
    fm_energy: np.ndarray     # (K,) -- 式 (11) 的 FM 能量/自旋

### 1.2 a–f：零温/有限温度方程、能量、稳定性与有限网络动力学

令 $x=m/\sqrt{2\alpha r}$。由论文式 (9)–(10) 可得 $m=\mathrm{erf}(x)$、$C=2x e^{-x^2}/(\sqrt{\pi}m)$、$r=(1-C)^{-2}$，再反推出 $\alpha=[m(1-C)]^2/(2x^2)$。这样可以一次生成 FM 的两条非零分支，不依赖容易漏根的逐点初值迭代。

In [ ]:
def a1_make_zero_temperature_parameter_grid(
    start: float = 0.02,
    stop: float = 20.0,
    points: int = 50_000,
) -> np.ndarray:
    """生成 x=m/sqrt(2*alpha*r) 的参数网格。"""
    # [输入] 小 x 覆盖低重叠支；大 x 覆盖 alpha -> 0 的高重叠支
    return np.linspace(start, stop, points)


def b1_solve_zero_temperature_rs(x: np.ndarray) -> ZeroTemperatureBranch:
    """按论文式 (9)–(11) 参数化求出零温 RS 的非零 FM 解。"""
    overlap = erf(x)
    # [推导] 论文式 (9)：m = erf[m/sqrt(2*r*alpha)]

    response = 2.0 * x * np.exp(-(x**2)) / (np.sqrt(np.pi) * overlap)
    # [中介变量] 由式 (10) 消去 alpha*r 后得到 C(x)

    noise = 1.0 / (1.0 - response) ** 2
    alpha = (overlap * (1.0 - response)) ** 2 / (2.0 * x**2)
    # [推导] 用 x 的定义反解载荷率 alpha；同一 alpha 可对应两条非零分支

    error_percent = 50.0 * (1.0 - overlap)
    fm_energy = 0.5 * alpha * (1.0 - noise) - 0.5 * overlap**2
    # [观测·数据] 错误率来自 overlap；能量逐项对应论文式 (11)

    return ZeroTemperatureBranch(
        alpha=alpha,
        overlap=overlap,
        response=response,
        noise=noise,
        error_percent=error_percent,
        fm_energy=fm_energy,
    )


def c1_split_fm_branches(
    branch: ZeroTemperatureBranch,
) -> tuple[np.ndarray, np.ndarray, int]:
    """返回低 m 支、高 m 支与二者合并处的索引。"""
    critical_index = int(np.argmax(branch.alpha))
    unstable_indices = np.arange(0, critical_index + 1)
    stable_indices = np.arange(critical_index, branch.alpha.size)
    # [判断] 论文指出两条解中较大的 m 对 m 扰动局部稳定
    return unstable_indices, stable_indices, critical_index


def d1_spin_glass_energy(alpha: np.ndarray) -> np.ndarray:
    """论文 p.1532 给出的 T=0 replica-symmetric SG 能量。"""
    alpha = np.asarray(alpha, dtype=float)
    # [推导] m=0 且 r=[1+sqrt(2/(pi*alpha))]^2，代入式 (11)
    # 得 E_SG=-1/pi-sqrt(2*alpha/pi)；根号内的分子分母不能写反
    return -1.0 / np.pi - np.sqrt(2.0 * alpha / np.pi)


def b2_asymptotic_error_count(N: np.ndarray, alpha: np.ndarray) -> np.ndarray:
    """论文式 (12)：小 alpha 下的平均错误自旋数。"""
    N = np.asarray(N, dtype=float)
    alpha = np.asarray(alpha, dtype=float)
    return N * np.sqrt(alpha / (2.0 * np.pi)) * np.exp(-1.0 / (2.0 * alpha))


def e1_linear_crossing(x: np.ndarray, y: np.ndarray) -> float:
    """在线性插值下返回 y 第一次跨过 0 的 x。"""
    crossing_indices = np.flatnonzero(y[:-1] * y[1:] <= 0.0)
    if crossing_indices.size == 0:
        raise ValueError("扫描区间内没有找到符号变化")
    index = int(crossing_indices[0])
    x0, x1 = map(float, x[index:index + 2])
    y0, y1 = map(float, y[index:index + 2])
    if y1 == y0:
        return 0.5 * (x0 + x1)
    # [数值稳定] 直接写割线公式，正穿越和负穿越都适用
    return x0 - y0 * (x1 - x0) / (y1 - y0)


def a3_make_independent_patterns(
    p: int,
    N: int,
    rng: np.random.Generator,
) -> np.ndarray:
    """生成 p 条相互独立的随机 ±1 模式。"""
    return rng.choice(np.array([-1, 1], dtype=np.int8), size=(p, N))


def b3_make_normalized_hebb_weights(patterns: np.ndarray) -> np.ndarray:
    """论文式 (2) 的 J_ij=(1/N) sum_mu xi_i^mu xi_j^mu。"""
    N = patterns.shape[1]
    weights = patterns.astype(float).T @ patterns.astype(float) / N
    # [存储] 先转 float 防止 p 较大时 int8 求和溢出
    np.fill_diagonal(weights, 0.0)
    # [约束] 论文 Hamiltonian 只对 i != j 求和，因此 J_ii=0
    return weights


def d3_run_zero_temperature_async(
    weights: np.ndarray,
    initial_state: np.ndarray,
    rng: np.random.Generator,
    max_sweeps: int = 20,
) -> tuple[np.ndarray, int, str]:
    """串行 heat-bath 在 T=0 的极限：每次只更新一个自旋。"""
    state = initial_state.copy().astype(np.int8)
    local_fields = weights @ state

    for sweep in range(1, max_sweeps + 1):
        flips = 0
        for neuron in rng.permutation(state.size):
            field = local_fields[neuron]
            new_value = 1 if field > 0 else -1 if field < 0 else int(state[neuron])
            # [动力学] T=0 时取局域场符号；零场保持原值，避免任意破坏平局

            if new_value != state[neuron]:
                change = new_value - int(state[neuron])
                state[neuron] = new_value
                local_fields += weights[:, neuron] * change
                flips += 1
                # [更新] 增量更新所有局域场，等价于重新计算 weights @ state

        if flips == 0:
            return state, sweep, "fixed"

    return state, max_sweeps, "max_sweeps"


def e4_overlap(state: np.ndarray, pattern: np.ndarray) -> float:
    """论文式 (3) 的有限 N 样本重叠。"""
    return float(np.mean(state.astype(float) * pattern.astype(float)))

## 2. 公共组件自检

这些检查先验证参数化方程确实落在论文报告的临界窗口，再运行后续绘图。它们不是把目标答案硬编码成曲线；曲线仍由式 (9)–(11) 独立计算。

In [ ]:
_x = a1_make_zero_temperature_parameter_grid()
_branch = b1_solve_zero_temperature_rs(_x)
_unstable_indices, _stable_indices, _critical_index = c1_split_fm_branches(_branch)

_alpha_c = float(_branch.alpha[_critical_index])
_m_c = float(_branch.overlap[_critical_index])
_error_c = float(_branch.error_percent[_critical_index])

assert np.all(np.isfinite(_branch.alpha))
assert 0.137 < _alpha_c < 0.139
assert 0.966 < _m_c < 0.968
assert 1.5 < _error_c < 1.8
# [判断] 分别对应论文的 alpha_c=0.138、m=0.967 与约1.5%错误

print(f"临界载荷 alpha_c = {_alpha_c:.5f}")
print(f"临界重叠 m_c = {_m_c:.5f}")
print(f"临界错误率 = {_error_c:.3f}%")

## 原文 Fig. 1 复现：零温 FM 分支与平均错误率

**原文定位**：p.1531，式 (9)–(11)；p.1532，Fig. 1。

**问题**：当存储模式数按 $p=\alpha N$ 线性增长时，和某一模式宏观相关的 FM（ferromagnetic，铁磁记忆态）还能存在到多大载荷？

**配方**：`a1 → b1 → c1 → e1 → f1`

**展示说明**：下面左右两个 panel 使用同一组 Fig. 1 数据；左侧补出完整双分支，右侧代替原文断轴放大稳定支。它们仍只对应一张论文图。

**论文数据边界**：原文给出 $\alpha_c=0.138$、临界处 $m=0.967$ 和约 1.5% 错误；Fig. 1 是平均场方程的理论曲线，不是带抽样误差的重复模拟。

In [ ]:
stable_order = _stable_indices[np.argsort(_branch.alpha[_stable_indices])]
unstable_order = _unstable_indices[np.argsort(_branch.alpha[_unstable_indices])]
# [观测·数据] 同一 alpha 下，高 m 支稳定、低 m 支不稳定

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(
    _branch.alpha[stable_order],
    _branch.error_percent[stable_order],
    label="稳定 FM（高 m）",
)
axes[0].plot(
    _branch.alpha[unstable_order],
    _branch.error_percent[unstable_order],
    linestyle="--",
    label="不稳定 FM（低 m）",
)
axes[0].axvline(_alpha_c, color="black", linestyle=":", label=rf"$\alpha_c={_alpha_c:.3f}$")
axes[0].set(
    xlim=(0.0, 0.155),
    ylim=(0.0, 52.0),
    xlabel="载荷率 α=p/N",
    ylabel="错误自旋比例（%）",
    title="完整非零 FM 分支",
)
axes[0].legend(fontsize=8)

axes[1].plot(
    _branch.alpha[stable_order],
    _branch.error_percent[stable_order],
    color="tab:blue",
)
axes[1].scatter([_alpha_c], [_error_c], color="tab:red", zorder=3)
axes[1].axvline(_alpha_c, color="black", linestyle=":")
axes[1].annotate(
    f"临界点\n({_alpha_c:.3f}, {_error_c:.2f}%)",
    xy=(_alpha_c, _error_c),
    xytext=(0.085, 1.25),
    arrowprops={"arrowstyle": "->"},
)
axes[1].set(
    xlim=(0.0, 0.145),
    ylim=(0.0, 2.0),
    xlabel="载荷率 α=p/N",
    ylabel="错误自旋比例（%）",
    title="论文 Fig. 1 的稳定支放大",
)

plt.tight_layout()
plt.show()

comparison = pd.DataFrame({
    "对照项": ["FM 消失点", "临界重叠", "临界错误率"],
    "Amit et al. 1985": ["alpha_c=0.138", "m=0.967", "约1.5%"],
    "本次数值解": [f"{_alpha_c:.5f}", f"{_m_c:.5f}", f"{_error_c:.3f}%"],
})
display(comparison)

**如何判断是否复现**：三项自检同时通过，并且稳定支在 $\alpha_c$ 处以非零 $m$ 终止，才复现了论文的一级式分支消失，而不是把它误画成从 $m=0$ 连续长出的二级相变。

**结果分析**：$p=\alpha N$ 可以随 $N$ 无限增长，但 FM 记忆态只在 $\alpha<\alpha_c$ 存在。临界处仍有很高重叠；“容量消失”指非零 FM 解整体消失，不是错误率缓慢爬到 50%。

**不能声称什么**：这张图不能证明 replica symmetry 在低温严格正确。论文自己指出低温 RS 会失稳，只认为容量与错误率的数值改动较小。

## 原文 Fig. 2 复现：有限温度相图

**原文定位**：p.1531 式 (5)–(8)，p.1532 Fig. 2，p.1533 式 (13)。

**问题**：随 $\alpha$ 增加，FM 记忆态在什么温度首次出现（$T_M$）、何时成为全局自由能最低态（$T_C$），以及 replica-symmetric FM 解何时失稳（$T_R$）？

**配方**：`a2(alpha,T 网格) → b2(式 6–8 鞍点) → c2(高 m 延续支) → d2(式 5 自由能 / 式 13 replicon) → e2(三个边界) → f2`

这里采用 Gauss–Hermite 求积计算高斯平均，并把 $C=\beta(1-q)$ 作为未知量。于是 $q=1-CT$、$r=q/(1-C)^2$，避免在低温直接求解高度病态的 $(q,r)$。

In [ ]:
# [数值积分] E_z[f(z)] = sum_j w_j f(sqrt(2)*x_j)/sqrt(pi)
_gh_nodes, _gh_weights = np.polynomial.hermite.hermgauss(80)
_normal_nodes = np.sqrt(2.0) * _gh_nodes
_normal_weights = _gh_weights / np.sqrt(np.pi)


def b4_finite_temperature_moments(
    m: float,
    C: float,
    alpha: float,
    temperature: float,
) -> dict[str, float]:
    """由单个宏观 overlap 的 RS 场计算式 (6)–(8) 所需高斯矩。"""
    q = 1.0 - C * temperature
    denominator = 1.0 - C
    if q <= 0.0 or denominator <= 0.0:
        raise ValueError("finite-temperature state left the physical q/C domain")

    r = q / denominator**2
    fields = np.sqrt(alpha * r) * _normal_nodes + m
    activations = np.tanh(fields / temperature)
    mean_tanh = float(np.dot(_normal_weights, activations))
    mean_tanh2 = float(np.dot(_normal_weights, activations**2))
    mean_sech4 = float(np.dot(_normal_weights, (1.0 - activations**2) ** 2))
    return {
        "m": float(m), "C": float(C), "q": float(q), "r": float(r),
        "mean_tanh": mean_tanh, "mean_tanh2": mean_tanh2,
        "mean_sech4": mean_sech4,
    }


def b5_solve_fm_state(
    alpha: float,
    temperature: float,
    initial: tuple[float, float],
) -> dict[str, float] | None:
    """求单一凝聚模式的非零 FM 鞍点；多初值中保留最大的 m 支。"""
    upper_C = min(0.999999, 0.999999 / temperature)

    def residual(values: np.ndarray) -> np.ndarray:
        moments = b4_finite_temperature_moments(
            float(values[0]), float(values[1]), alpha, temperature
        )
        return np.array([
            values[0] - moments["mean_tanh"],
            moments["q"] - moments["mean_tanh2"],
        ])

    candidates = [initial, (0.98, min(initial[1], upper_C)), (0.65, 0.5)]
    solutions = []
    for guess_m, guess_C in candidates:
        fit = least_squares(
            residual,
            x0=np.array([
                np.clip(guess_m, 1e-5, 0.999999),
                np.clip(guess_C, 1e-6, upper_C),
            ]),
            bounds=(np.array([1e-8, 1e-8]), np.array([0.999999, upper_C])),
            xtol=1e-11, ftol=1e-11, gtol=1e-11, max_nfev=300,
        )
        residual_norm = float(np.max(np.abs(fit.fun)))
        if fit.success and residual_norm < 1e-7 and fit.x[0] > 1e-3:
            state = b4_finite_temperature_moments(
                float(fit.x[0]), float(fit.x[1]), alpha, temperature
            )
            state["residual"] = residual_norm
            solutions.append(state)

    return max(solutions, key=lambda state: state["m"], default=None)


def b6_solve_sg_state(
    alpha: float,
    temperature: float,
    initial_C: float,
) -> dict[str, float] | None:
    """在 m=0 子空间求 SG 鞍点。"""
    upper_C = min(0.999999, 0.999999 / temperature)

    def residual(values: np.ndarray) -> np.ndarray:
        moments = b4_finite_temperature_moments(
            0.0, float(values[0]), alpha, temperature
        )
        return np.array([moments["q"] - moments["mean_tanh2"]])

    fit = least_squares(
        residual,
        x0=np.array([np.clip(initial_C, 1e-6, upper_C)]),
        bounds=(np.array([1e-8]), np.array([upper_C])),
        xtol=1e-11, ftol=1e-11, gtol=1e-11, max_nfev=300,
    )
    residual_norm = float(np.max(np.abs(fit.fun)))
    if not fit.success or residual_norm >= 1e-7:
        return None
    state = b4_finite_temperature_moments(
        0.0, float(fit.x[0]), alpha, temperature
    )
    state["residual"] = residual_norm
    return state


def d4_rs_free_energy(
    state: dict[str, float], alpha: float, temperature: float
) -> float:
    """逐项实现论文式 (5) 的 replica-symmetric 自由能/自旋。"""
    m, C, q, r = state["m"], state["C"], state["q"], state["r"]
    fields = np.sqrt(alpha * r) * _normal_nodes + m
    log_two_cosh = np.logaddexp(fields / temperature, -fields / temperature)
    bracket = (
        temperature * np.log(1.0 - C)
        + (temperature - 1.0) * C / (1.0 - C)
        + r * C
    )
    return float(
        0.5 * m**2
        + 0.5 * alpha * bracket
        - temperature * np.dot(_normal_weights, log_two_cosh)
    )


def e5_replicon_eigenvalue(
    state: dict[str, float], alpha: float, temperature: float
) -> float:
    """论文式 (13)；lambda=0 给出 RS FM 的低温失稳线 T_R。"""
    beta = 1.0 / temperature
    return float(
        (1.0 - state["C"]) ** 2
        - alpha * beta**2 * state["mean_sech4"]
    )


def e6_first_crossing(x: np.ndarray, y: np.ndarray) -> float:
    """忽略 NaN 后寻找第一处符号变化，并用割线做局部插值。"""
    valid = np.isfinite(x) & np.isfinite(y)
    x_valid, y_valid = x[valid], y[valid]
    if x_valid.size < 2:
        return float("nan")
    indices = np.flatnonzero(y_valid[:-1] * y_valid[1:] <= 0.0)
    if indices.size == 0:
        return float("nan")
    index = int(indices[0])
    x0, x1 = x_valid[index:index + 2]
    y0, y1 = y_valid[index:index + 2]
    return float(x0 - y0 * (x1 - x0) / (y1 - y0))


def a2_zero_temperature_guess(alpha: float) -> tuple[float, float]:
    """从已求出的稳定零温分支插值得到有限温度延续的初值。"""
    return (
        float(np.interp(alpha, _branch.alpha[stable_order], _branch.overlap[stable_order])),
        float(np.interp(alpha, _branch.alpha[stable_order], _branch.response[stable_order])),
    )


def run_paper_fig2_phase_scan(
    alpha_grid: np.ndarray,
    temperature_grid: np.ndarray,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    """沿温度延续高-m FM 支，并提取 T_M、T_C 与 T_R。"""
    states = []
    boundaries = []
    for alpha in alpha_grid:
        fm_initial = a2_zero_temperature_guess(float(alpha))
        zero_sg_scale = np.sqrt(2.0 / (np.pi * alpha))
        sg_initial_C = zero_sg_scale / (1.0 + zero_sg_scale)
        alpha_rows = []

        for temperature in temperature_grid:
            fm_state = b5_solve_fm_state(
                float(alpha), float(temperature), fm_initial
            )
            if fm_state is None:
                continue

            fm_initial = (fm_state["m"], fm_state["C"])
            sg_state = b6_solve_sg_state(
                float(alpha), float(temperature), sg_initial_C
            )
            if sg_state is not None:
                sg_initial_C = sg_state["C"]

            fm_free_energy = d4_rs_free_energy(fm_state, alpha, temperature)
            sg_free_energy = (
                d4_rs_free_energy(sg_state, alpha, temperature)
                if sg_state is not None else float("nan")
            )
            row = {
                "alpha": float(alpha),
                "temperature": float(temperature),
                "m": fm_state["m"],
                "q": fm_state["q"],
                "r": fm_state["r"],
                "fm_residual": fm_state["residual"],
                "fm_free_energy": fm_free_energy,
                "sg_free_energy": sg_free_energy,
                "free_energy_difference": fm_free_energy - sg_free_energy,
                "replicon": e5_replicon_eigenvalue(
                    fm_state, float(alpha), float(temperature)
                ),
            }
            states.append(row)
            alpha_rows.append(row)

        frame = pd.DataFrame(alpha_rows).sort_values("temperature")
        if frame.empty:
            boundaries.append({
                "alpha": float(alpha), "T_M": np.nan,
                "T_C": np.nan, "T_R": np.nan,
            })
            continue

        boundaries.append({
            "alpha": float(alpha),
            "T_M": float(frame["temperature"].max()),
            "T_C": e6_first_crossing(
                frame["temperature"].to_numpy(),
                frame["free_energy_difference"].to_numpy(),
            ),
            "T_R": e6_first_crossing(
                frame["temperature"].to_numpy(),
                frame["replicon"].to_numpy(),
            ),
        })

    return pd.DataFrame(states), pd.DataFrame(boundaries)


# [实验控制] 低温区用对数网格解析 T_R，主相图用线性网格解析 T_C/T_M
phase_alpha = np.unique(np.concatenate([
    np.linspace(0.002, 0.050, 25),
    np.array([0.051]),
    np.linspace(0.055, 0.137, 30),
]))
phase_temperature = np.unique(np.concatenate([
    np.geomspace(1e-4, 0.03, 60),
    np.linspace(0.035, 1.02, 198),
]))

fig2_states, fig2_boundaries = run_paper_fig2_phase_scan(
    phase_alpha, phase_temperature
)

# [失败边界] 不用论文数字硬填缺失点；求解失败数直接展示给读者
fig2_gate = pd.Series({
    "alpha 网格数": len(phase_alpha),
    "T_M 有效点": int(fig2_boundaries["T_M"].notna().sum()),
    "T_C 有效点": int(fig2_boundaries["T_C"].notna().sum()),
    "T_R 有效点": int(fig2_boundaries["T_R"].notna().sum()),
    "最大 FM 方程残差": float(fig2_states["fm_residual"].max()),
})

fig, axis = plt.subplots(figsize=(8.2, 5.2))
axis.plot(fig2_boundaries["alpha"], fig2_boundaries["T_M"], label=r"$T_M$：FM 出现")
axis.plot(fig2_boundaries["alpha"], fig2_boundaries["T_C"], label=r"$T_C$：FM/SG 自由能交叉")
axis.set(
    xlim=(0.0, 0.145), ylim=(0.0, 1.03),
    xlabel=r"载荷率 $\alpha=p/N$", ylabel="温度 T",
    title="论文 Fig. 2：FM 临界温度相图",
)
axis.axvline(_alpha_c, color="black", linestyle=":", linewidth=1)
axis.legend(loc="upper right")

inset = axis.inset_axes([0.53, 0.52, 0.40, 0.32])
inset.plot(fig2_boundaries["alpha"], fig2_boundaries["T_R"], color="tab:green")
inset.set(xlim=(0.045, 0.14), ylim=(0.0, 0.022), title=r"$T_R$ 放大")
inset.tick_params(labelsize=7)
plt.tight_layout()
plt.show()

display(fig2_gate.to_frame("本次数值解"))
display(fig2_boundaries.head())

**如何判断是否复现**：$T_M$ 应从 $\alpha\to0$ 时接近 1，下降到 $\alpha_c\approx0.138$ 附近约 0.07；$T_C$ 应从 $(0,1)$ 下降并在 $\alpha\approx0.051$ 附近到达零温；$T_R$ 位于很低温区，因此单独放大到 $T<0.02$。同时检查表中的有效点数和最大鞍点残差，不能只看曲线像不像。

**结果分析**：$T_M$ 回答记忆态是否存在，$T_C$ 回答它是否是全局自由能最低态，$T_R$ 回答 RS 描述本身是否稳定。三条线测的是三个不同问题，不能合成一个“容量”。

**不能声称什么**：这是 replica-symmetric 方程的数值重画，不是 Monte Carlo 验证；论文说明低温 replica-symmetry breaking 会修正 $T_C$ 和混合态边界。

## 原文公式核对 A：记忆态“存在”不等于“是基态”

**原文定位**：p.1531 式 (11)；p.1532 左栏能量比较。

**问题**：为什么论文同时给出 $0.138$ 和 $0.051$ 两个容量数字？

**配方**：`实验1稳定支 → d1 FM能量 / d2 SG能量 → e2 交叉点 → f1`

这里比较的是两类 RS 解的能量。$\alpha_c$ 回答“FM 局部稳定解是否还存在”；能量交叉回答“FM 是否低于 SG（spin glass，自旋玻璃伪态）而成为全局基态”。

In [ ]:
energy_alpha = _branch.alpha[stable_order]
energy_fm = _branch.fm_energy[stable_order]
energy_sg = d1_spin_glass_energy(energy_alpha)
energy_difference = energy_fm - energy_sg
alpha_ground_state = e1_linear_crossing(energy_alpha, energy_difference)
# [观测·数据] 差值从负变正：FM 从全局较低能变成仅局部稳定

if not np.isclose(alpha_ground_state, 0.051, atol=0.002):
    raise AssertionError(
        f"FM/SG 能量交叉为 {alpha_ground_state:.6f}，"
        "未落在论文 0.051 的数值容差内；请优先检查 SG 能量公式。"
    )

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(energy_alpha, energy_fm, label="FM 记忆态")
axes[0].plot(energy_alpha, energy_sg, label="SG 伪态")
axes[0].axvline(
    alpha_ground_state,
    color="black",
    linestyle=":",
    label=rf"能量交叉 $\alpha={alpha_ground_state:.3f}$",
)
axes[0].axvline(_alpha_c, color="tab:red", linestyle="--", label=rf"FM消失 $\alpha_c={_alpha_c:.3f}$")
axes[0].set(
    xlim=(0.0, 0.145),
    xlabel="载荷率 α=p/N",
    ylabel="每个自旋的能量",
    title="FM 与 SG 的零温能量",
)
axes[0].legend(fontsize=8)

axes[1].plot(energy_alpha, energy_difference, color="tab:purple")
axes[1].axhline(0.0, color="black", linewidth=1)
axes[1].axvspan(0.0, alpha_ground_state, alpha=0.15, color="tab:green", label="FM 为全局基态")
axes[1].axvspan(alpha_ground_state, _alpha_c, alpha=0.15, color="tab:orange", label="FM 仅为亚稳态")
axes[1].set(
    xlim=(0.0, 0.145),
    xlabel="载荷率 α=p/N",
    ylabel="E_FM - E_SG",
    title="两个容量阈值对应不同问题",
)
axes[1].legend(fontsize=8)

plt.tight_layout()
plt.show()

critical_energy = float(_branch.fm_energy[_critical_index])
print(f"FM/SG 能量交叉 alpha = {alpha_ground_state:.5f}（论文：0.051）")
print(f"alpha_c 处 FM 能量 = {critical_energy:.4f}（论文：-0.5014）")

**如何判断是否复现**：能量交叉应落在 $0.050<\alpha<0.052$，且 $\alpha_c$ 处的 FM 能量接近 $-0.5014$。

**结果分析**：$\alpha<0.051$ 时，FM 是全局基态；$0.051<\alpha<0.138$ 时，它仍是可检索的局部稳定记忆态，但 SG 的能量更低。必须给出与模式相关的初始提示，不能期待系统从任意状态自动找到记忆。

**不能声称什么**：`0.051` 不是通常所说的提示召回容量；把它直接替换 `0.138` 会混淆全局平衡与局部吸引盆。

## 原文公式核对 B：允许小错误与要求零错误是两种容量

**原文定位**：p.1531 式 (12)；p.1532 Fig. 1 下方。

**问题**：Hopfield 的线性容量与 Weisbuch/Posner 的 $p<N/(2\ln N)$ 为什么不矛盾？

**配方**：`a2 N网格 → b2 式(12) → e3 幂律斜率 → f1`

令 $\alpha=(2\kappa\ln N)^{-1}$。式 (12) 的主导幂次为 $N_e\sim N^{1-\kappa}$；缓慢变化的 $1/\sqrt{\ln N}$ 前因子仍保留在本次数值曲线中。

In [ ]:
N_values = np.logspace(2, 12, 240)
kappa_values = [0.75, 1.0, 1.25]
rows = []

fig, ax = plt.subplots(figsize=(8, 5))

for kappa in kappa_values:
    alpha_values = 1.0 / (2.0 * kappa * np.log(N_values))
    error_counts = b2_asymptotic_error_count(N_values, alpha_values)
    # [实验控制] 三条曲线只改变 kappa；N 网格和式 (12) 完全相同

    fitted_slope = float(np.polyfit(
        np.log(N_values[-100:]),
        np.log(error_counts[-100:]),
        deg=1,
    )[0])
    # [观测·数据] 有限区间斜率会比 1-kappa 略低，因为保留了对数前因子

    rows.append({
        "kappa": kappa,
        "主导幂次 1-kappa": 1.0 - kappa,
        "末段拟合斜率": fitted_slope,
        "N=10^12 时平均错误数": float(error_counts[-1]),
    })
    ax.loglog(N_values, error_counts, label=rf"$\kappa={kappa}$")

ax.axhline(1.0, color="black", linestyle=":", label="平均1个错误")
ax.set(
    xlabel="网络规模 N",
    ylabel="式 (12) 的平均错误自旋数 N_e",
    title="alpha=(2 kappa ln N)^(-1) 下的精确存储尺度",
)
ax.legend()
plt.tight_layout()
plt.show()

scaling_results = pd.DataFrame(rows)
display(scaling_results.round(4))

assert scaling_results.loc[0, "末段拟合斜率"] > 0.0
assert scaling_results.loc[2, "末段拟合斜率"] < 0.0
# [判断] kappa<1 时错误总数增长；kappa>1 时错误总数消失

**如何判断是否复现**：$\kappa<1$ 的曲线最终上升，$\kappa>1$ 的曲线最终下降；拟合斜率应接近但略低于 $1-\kappa$。

**结果分析**：固定 $\alpha<0.138$ 可以让错误比例很小，同时存储 $p=\alpha N$ 条模式；但若要求整个长度为 $N$ 的模式一个 bit 都不错，$\alpha$ 必须随 $N$ 收缩到约 $1/(2\ln N)$。两种结论测量的是不同精度标准。

**不能声称什么**：平均错误数小于 1 不等于每次试验都零错误；式 (12) 给的是渐近平均量，不是零错误概率的完整分布。

## 新增机制检查：有限 N 的零温异步动力学（非论文图）

**为什么补这个实验**：前面直接求解 $N\to\infty$ 的 RS 方程。这里回到论文式 (1)–(3) 的有限网络，从一条已存模式出发，检查零温串行 heat-bath 动力学最后还保留多少重叠。

**配方**：`a3 → b3 → c2模式初态 → d3 → e4 → f1`

扫描 $N=300$、七个载荷；每个载荷生成 6 个独立网络，每个网络抽 8 条模式。理论曲线与有限网络共享 $\alpha$，但不是把理论解强塞进动力学。

In [ ]:
rng = np.random.default_rng(SEED)
N = 300
alpha_values = np.array([0.03, 0.06, 0.10, 0.13, 0.14, 0.16, 0.20])
networks_per_alpha = 6
cues_per_network = 8
max_sweeps = 20
# *[输入] 小型 Colab 扫描；样本量用于机制检查，不用于重新估计热力学临界点

rows = []

for alpha in alpha_values:
    overlaps = []
    fixed_flags = []

    for _ in range(networks_per_alpha):
        p = max(1, int(round(alpha * N)))
        patterns = a3_make_independent_patterns(p=p, N=N, rng=rng)
        weights = b3_make_normalized_hebb_weights(patterns)
        # [存储] 每个网络重新采样模式，避免把单个容易/困难实例当成总体结论

        cue_indices = rng.choice(p, size=min(cues_per_network, p), replace=False)
        for cue_index in cue_indices:
            initial_state = patterns[cue_index].copy()
            # [输入] 论文将记忆诊断为外部刺激触发后的动态持久性；这里从精确模式启动

            final_state, _, status = d3_run_zero_temperature_async(
                weights,
                initial_state,
                rng,
                max_sweeps=max_sweeps,
            )
            overlaps.append(e4_overlap(final_state, initial_state))
            fixed_flags.append(status == "fixed")

    overlaps = np.asarray(overlaps, dtype=float)
    mean_overlap = float(np.mean(overlaps))
    ci95 = 1.96 * float(np.std(overlaps, ddof=1)) / np.sqrt(overlaps.size)

    rows.append({
        "alpha": float(alpha),
        "p": int(round(alpha * N)),
        "轨迹数": int(overlaps.size),
        "平均终态重叠": mean_overlap,
        "95% CI 半宽": ci95,
        "m>=0.9 比例": float(np.mean(overlaps >= 0.9)),
        "20轮内固定比例": float(np.mean(fixed_flags)),
    })

finite_results = pd.DataFrame(rows)

fig, ax = plt.subplots(figsize=(8, 5))
ax.errorbar(
    finite_results["alpha"],
    finite_results["平均终态重叠"],
    yerr=finite_results["95% CI 半宽"],
    marker="o",
    capsize=4,
    label=f"有限网络 N={N}",
)

theory_mask = energy_alpha <= _alpha_c
ax.plot(
    energy_alpha[theory_mask],
    _branch.overlap[stable_order][theory_mask],
    color="black",
    linestyle="--",
    label="N→∞ RS 稳定 FM",
)
ax.axvline(_alpha_c, color="tab:red", linestyle=":", label=rf"论文 $\alpha_c={_alpha_c:.3f}$")
ax.set(
    xlim=(0.02, 0.205),
    ylim=(0.0, 1.02),
    xlabel="载荷率 α=p/N",
    ylabel="终态与起始模式的重叠 m",
    title="有限 N 动力学与热力学稳定支",
)
ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

display_frame = finite_results.copy()
for column in ["m>=0.9 比例", "20轮内固定比例"]:
    display_frame[column] = display_frame[column].map(lambda value: f"{value:.1%}")
display(display_frame.round({"平均终态重叠": 3, "95% CI 半宽": 3}))

**如何判断这项机制检查**：低载荷的平均终态重叠应接近 1；接近和超过 $\alpha_c$ 后，有限样本会出现更大的网络间波动与更多低重叠终态。图中理论线是热力学稳定 FM 分支，散点是有限网络轨迹，二者不要求逐点重合。

**结果分析**：若运行结果呈上述方向，它说明平均场分支的容量压力在有限网络动力学中可见；它不构成对 $0.138$ 的独立高精度估计。

**不能声称什么**：本节没有复现论文 Fig. 2 的完整有限温度相图，也没有检验 replica-symmetry breaking。`m>=0.9` 是本节显式增加的可读性阈值，不是论文定义的临界判据。

## 3. 最终证据边界

运行全部单元后，可以声称：零温 RS 方程重画了原文 Fig. 1，并复核 $\alpha_c\approx0.138$、$m_c\approx0.967$ 与临界错误率；有限温度 RS 鞍点、自由能和 replicon 条件重画了原文 Fig. 2 的 $T_M/T_C/T_R$；另外核对了零温 FM/SG 能量交叉与式 (12) 的精确存储尺度。

有限 $N$ 异步动力学仍只是本 notebook 新增的方向性机制检查。不能声称它是第三张论文图，也不能据此直接验证 replica 方法或证明低温 RS 严格成立。